In [ ]:
import json

In [ ]:
import pickle

In [ ]:
import tqdm

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
import pandas as pd

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

In [ ]:
df = pd.read_excel("data_clean.xlsx", header=0)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
sum(df["title_rus"].isna())

In [ ]:
df = df.fillna("")

In [ ]:
df[df.duplicated(subset=["title_rus"])]

In [ ]:
df.shape

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can extract short and concise tags for student projects title in russian.
Please, for a given student project title return a list of corresponding tags, which depicts the field of science to which the project is relevant.
For example, for a project title "Рекомендательная система для студенческих проектов" you answer might be the following list of strings with underscores.
["machine_learning", "recommender_systems", "software_engineering"]
Also another example, for a project title "Прогнозирование эффектов процентной политики Банка России" you answer might be the following list of strings with underscores.
["economics", "macroeconomics", "monetary_policy", "forecasting", "time_series"]
Return strictly a list of strings as an answer.
"""

In [ ]:
messages = []

In [ ]:
for row in df.iterrows():
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row[1]["title_rus"]},
        )
    )

In [ ]:
tags_set = set()

In [ ]:
titles_with_tags_dict = dict()

In [ ]:
for message in tqdm.tqdm(messages):
    text = processor.apply_chat_template(
        message, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(**inputs, max_new_tokens=1024)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    answer = json.loads(processor.parse_response(response)["content"])

    titles_with_tags_dict[message[1]["content"]] = answer
    tags_set = tags_set.union(answer)

In [ ]:
with open("titles_with_tags_dict.pkl", "wb") as f:
    pickle.dump(titles_with_tags_dict, f)

In [ ]:
with open("tags_set.pkl", "wb") as f:
    pickle.dump(tags_set, f)

In [ ]:
len(tags_set)

In [ ]:
list(titles_with_tags_dict.values())[0]

In [ ]:
list(titles_with_tags_dict.values())[-1]